# NC Capstone: A.I. Business Sentiment and Employment by State
This notebook loads and combines BLS and BTOS datasets using relative paths, then seeks to analyze the relationship between A.I. sentiment and employment by type and U.S. State over time. 

---

Load Libraries

In [1]:
import pandas as pd
from glob import glob
import os

Checking to Make Sure File Path is Working

In [2]:
bls_paths = sorted(glob("data/state_M20*_dl.xlsx"))
print("Files found:")
print(bls_paths)

Files found:
['data\\state_M2020_dl.xlsx', 'data\\state_M2021_dl.xlsx', 'data\\state_M2022_dl.xlsx', 'data\\state_M2023_dl.xlsx', 'data\\state_M2024_dl.xlsx']


Load and Combine BLS Data
This block loads all BLS Excel files from the `data/` folder.

In [3]:

bls_dfs = []

for path in bls_paths:
    try:
        df = pd.read_excel(path)
        df["source_file"] = os.path.basename(path)
        bls_dfs.append(df)
    except Exception as e:
        print(f"Failed to load {path}: {e}")

# Combine all BLS data
bls_combined = pd.concat(bls_dfs, ignore_index=True)
print("\n BLS data combined. Sample:")
display(bls_combined.head())


 BLS data combined. Sample:


,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY,source_file,PCT_RPT
0,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,41.07,18690,24060,36250,56980,85430,NaN,NaN,state_M2020_dl.xlsx,NaN
1,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,91.89,47740,67330,95120,134320,191130,NaN,NaN,state_M2020_dl.xlsx,NaN
2,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,#,49480,97930,161290,#,#,NaN,NaN,state_M2020_dl.xlsx,NaN
3,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,#,48030,67740,101170,153050,#,NaN,NaN,state_M2020_dl.xlsx,NaN
4,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1031,Legislators,...,*,16220,17190,18820,27920,55970,True,NaN,state_M2020_dl.xlsx,NaN


BTOS Collection Date Translation Added Here

In [25]:
date_translation_path = "data/btos_collection_dates.csv"
try:
    date_translation = pd.read_csv(date_translation_path)
    print("BTOS date translation loaded successfully.")
    display(date_translation.head())
except Exception as e:
    print(f"Failed to load {date_translation_path}: {e}")

BTOS date translation loaded successfully.


,Smpdt,Col Start,Col End,Ref Start,Ref End
0,202215,07/18/2022,07/31/2022,07/04/2022,07/17/2022
1,202216,08/01/2022,08/14/2022,07/18/2022,07/31/2022
2,202217,08/15/2022,08/28/2022,08/01/2022,08/14/2022
3,202218,08/29/2022,09/11/2022,08/15/2022,08/28/2022
4,202219,09/12/2022,09/25/2022,08/29/2022,09/11/2022


Load and Combine BTOS Data.  
This block loads the BTOS survey data files from the `data/` folder.

In [4]:
btos_paths = ["data/State.xlsx", "data/State_v1.xlsx"]
btos_dfs = []

for path in btos_paths:
    try:
        df = pd.read_excel(path)
        df["source_file"] = os.path.basename(path)
        btos_dfs.append(df)
    except Exception as e:
        print(f"Failed to load {path}: {e}")

# Combine all BTOS data
btos_combined = pd.concat(btos_dfs, ignore_index=True)
print("\n BTOS data combined. Sample:")
display(btos_combined.head())


 BTOS data combined. Sample:


,State,Question ID,Question,Answer ID,Answer,202512,202511,202510,202509,202508,...,202224,202223,202222,202221,202220,202219,202218,202217,202216,202215
0,AK,2.0,"Overall, how would you describe this business'...",1.0,Excellent,S,S,S,S,S,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AK,2.0,"Overall, how would you describe this business'...",2.0,Above average,17.9%,16.5%,13.9%,19.6%,21.4%,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AK,2.0,"Overall, how would you describe this business'...",3.0,Average,46.2%,52.9%,69.6%,34.6%,57.1%,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AK,2.0,"Overall, how would you describe this business'...",4.0,Below average,18.2%,18.2%,S,32.1%,S,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AK,2.0,"Overall, how would you describe this business'...",5.0,Poor,S,S,S,S,S,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Renaming BTOS Date Columns to Make them More Usable

In [ ]:

collection_dates = pd.read_csv(date_translation_path)
print("BTOS collection dates loaded successfully.")
display(collection_dates.head())

# Convert Smpdt to string and create mapping
collection_dates['Smpdt'] = collection_dates['Smpdt'].astype(str)
date_mapping = dict(zip(collection_dates['Smpdt'], collection_dates['Ref End']))

# Check current BTOS columns
print("Current BTOS date columns:", [col for col in btos_combined.columns if str(col).startswith('202')][:10])

# Find matching columns and rename
date_columns = [col for col in btos_combined.columns if str(col) in date_mapping.keys()]
print(f"Found {len(date_columns)} matching columns to rename")

if len(date_columns) > 0:
    rename_dict = {col: date_mapping[str(col)] for col in date_columns}
    btos_combined = btos_combined.rename(columns=rename_dict)
    print(f"Successfully renamed {len(rename_dict)} date columns to Ref End dates.")
    
    # Verify the renaming worked
    print("Sample new column names:", list(btos_combined.columns)[-10:])
else:
    print("No matching columns found")

BTOS collection dates loaded successfully.


,Smpdt,Col Start,Col End,Ref Start,Ref End
0,202215,07/18/2022,07/31/2022,07/04/2022,07/17/2022
1,202216,08/01/2022,08/14/2022,07/18/2022,07/31/2022
2,202217,08/15/2022,08/28/2022,08/01/2022,08/14/2022
3,202218,08/29/2022,09/11/2022,08/15/2022,08/28/2022
4,202219,09/12/2022,09/25/2022,08/29/2022,09/11/2022


All BTOS columns:
['State', 'Question_ID', 'Question', 'Answer_ID', 'Answer', '06/01/2025', '05/18/2025', '05/04/2025', '04/20/2025', '04/06/2025', '03/23/2025', '03/09/2025', '02/23/2025', '02/09/2025', '01/26/2025', '01/12/2025', '12/29/2024', '12/15/2024', '12/01/2024', '11/17/2024', '11/03/2024', '10/20/2024', '10/06/2024', '09/22/2024', '09/08/2024', '08/25/2024', '08/11/2024', '07/28/2024', '07/14/2024', '06/30/2024', '06/16/2024', '06/02/2024', '05/19/2024', '05/05/2024', '04/21/2024', '04/07/2024', '03/24/2024', '03/10/2024', '02/25/2024', '02/11/2024', '01/28/2024', '01/14/2024', '12/31/2023', '12/17/2023', '12/03/2023', '11/19/2023', '11/05/2023', '10/22/2023', '10/08/2023', '09/24/2023', '09/10/2023', 'source_file']

Total columns: 52

Columns that look like dates: ['06/01/2025', '05/18/2025', '05/04/2025', '04/20/2025', '04/06/2025', '03/23/2025', '03/09/2025', '02/23/2025', '02/09/2025', '01/26/2025', '01/12/2025', '12/29/2024', '12/15/2024', '12/01/2024', '11/17/2024', '1

Save Combined Outputs

In [ ]:
bls_combined.to_csv("data/combined_bls.csv", index=False)
btos_combined.to_csv("data/combined_btos.csv", index=False)
print("\n Output saved: data/combined_bls.csv and data/combined_btos.csv")


 Output saved: data/combined_bls.csv and data/combined_btos.csv
Saved BTOS columns (first 10): ['State', 'Question_ID', 'Question', 'Answer_ID', 'Answer', '06/01/2025', '05/18/2025', '05/04/2025', '04/20/2025', '04/06/2025']


Combined Staging Data Preview

In [41]:
print("\nPreview: BLS Combined Data")
display(bls_combined.head())

print("\nPreview: BTOS Combined Data")
display(btos_combined.head())


Preview: BLS Combined Data


,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY,source_file,PCT_RPT
0,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,41.07,18690,24060,36250,56980,85430,NaN,NaN,state_M2020_dl.xlsx,NaN
1,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,91.89,47740,67330,95120,134320,191130,NaN,NaN,state_M2020_dl.xlsx,NaN
2,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,#,49480,97930,161290,#,#,NaN,NaN,state_M2020_dl.xlsx,NaN
3,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,#,48030,67740,101170,153050,#,NaN,NaN,state_M2020_dl.xlsx,NaN
4,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1031,Legislators,...,*,16220,17190,18820,27920,55970,True,NaN,state_M2020_dl.xlsx,NaN



Preview: BTOS Combined Data


,State,Question_ID,Question,Answer_ID,Answer,06/01/2025,05/18/2025,05/04/2025,04/20/2025,04/06/2025,...,12/31/2023,12/17/2023,12/03/2023,11/19/2023,11/05/2023,10/22/2023,10/08/2023,09/24/2023,09/10/2023,source_file
0,AK,2.0,"Overall, how would you describe this business'...",1.0,Excellent,S,S,S,S,S,...,16.7%,17.9%,15.2%,14.0%,13.7%,15.8%,22.3%,16.2%,23.0%,State.xlsx
1,AK,2.0,"Overall, how would you describe this business'...",2.0,Above average,17.9%,16.5%,13.9%,19.6%,21.4%,...,24.9%,17.0%,24.6%,24.5%,21.1%,20.7%,25.1%,14.9%,18.2%,State.xlsx
2,AK,2.0,"Overall, how would you describe this business'...",3.0,Average,46.2%,52.9%,69.6%,34.6%,57.1%,...,43.7%,45.3%,46.6%,42.8%,55.4%,45.8%,38.6%,49.6%,42.9%,State.xlsx
3,AK,2.0,"Overall, how would you describe this business'...",4.0,Below average,18.2%,18.2%,S,32.1%,S,...,S,12.2%,11.3%,17.1%,S,13.0%,S,12.6%,15.9%,State.xlsx
4,AK,2.0,"Overall, how would you describe this business'...",5.0,Poor,S,S,S,S,S,...,S,S,S,S,S,S,S,S,S,State.xlsx


BTOS Data Cleaning Begins in Earnest Here

In [42]:
from pandasql import sqldf

btos_combined.columns = btos_combined.columns.str.replace(' ', '_')
btos_combined = btos_combined.loc[:, ~btos_combined.columns.isna()]

# I am only using the "State" source - which goes as far back as Sept 2023, 
# (cont'd) because prior to Sept 2023 (State v1 file) there was not an A.I. question in the data.

query = """
SELECT distinct Question, Question_ID
FROM btos_combined
WHERE 1=1
and Question like '%Artificial Intelligence%'
and source_file in ('State.xlsx')
limit 10
"""
print("\nQuerying BTOS Combined Data for AI-related questions:")


result = sqldf(query, locals())


print(result)


Querying BTOS Combined Data for AI-related questions:
                                            Question  Question_ID
0  In the last two weeks, did this business use A...          7.0
1  During the next six months, do you think this ...         26.0


Getting Cleaned BTOS Dataset Together